In [ ]:
# 1. CẤU HÌNH VÀ HÀM CHẠY LỆNH
# Chạy lần lượt từ trên xuống. Chỉ một tiến trình train trong runtime.

import os, sys, json, subprocess, hashlib
from pathlib import Path
ROOT = Path('/content/career_tf_fresh')
PY = '/content/career_tf_env/bin/python'
COMMIT = '550065adb0f693dee0d2fbb19419e00d74500cae'
os.environ.update(KERAS_BACKEND='tensorflow', MPLBACKEND='Agg', PYTHONPATH=str(ROOT))
def active_training():
    lines = subprocess.check_output(['ps','-eo','pid,args'],text=True).splitlines()
    return [line for line in lines if ' -m career_auto.training ' in line]
def shell(cmd, cwd=None):
    print('RUN:', ' '.join(map(str,cmd)), flush=True)
    logdir = ROOT/'outputs/colab_logs'
    logdir.mkdir(parents=True,exist_ok=True)
    import datetime
    logpath = logdir/(datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')+'.log')
    with logpath.open('w') as log:
        p=subprocess.Popen(list(map(str,cmd)),cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for line in p.stdout:
            print(line,end='',flush=True); log.write(line); log.flush()
        result=p.wait()
    if result: raise RuntimeError(f'Exit {result}. Xem log: {logpath}')
def train(config,run,extra=()):
    assert not active_training(), 'Đã có tiến trình train; không chạy trùng!'
    shell([PY,'-u','-m','career_auto.training','--config',config,'--run',run,*extra],ROOT)
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
print('Workspace:',ROOT)
print('Tiến trình train:',active_training())

print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True))


In [ ]:
# 2. SOURCE CÔNG KHAI VÀ MÔI TRƯỜNG LINUX RIÊNG
# Không upload file local, không dùng checkpoint Mac.
assert not active_training(), 'Chờ train hiện tại kết thúc trước khi sửa môi trường'
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/theihoz/ML.git',str(ROOT)],check=True)
shell(['git','checkout','--detach',COMMIT],ROOT)
assert not (ROOT/'outputs/auto_dev/state.json').exists(), 'Không chạy coordinator Mac ở Colab'
subprocess.run([sys.executable,'-m','pip','install','virtualenv'],check=True)
subprocess.run([sys.executable,'-m','virtualenv','/content/career_tf_env'],check=True)
shell([PY,'-m','pip','install','tensorflow[and-cuda]==2.20.0','tensorflow-text==2.20.1','keras==3.15.1','keras-hub==0.32.0','numpy==2.4.6','pandas==3.1.0','matplotlib','seaborn','openpyxl','scikit-learn','nbformat','nbclient','ipykernel','pyarrow','huggingface-hub'])
with (ROOT/'outputs/colab_environment_freeze.txt').open('w') as f:
    subprocess.run([PY,'-m','pip','freeze'],stdout=f,check=True)


In [ ]:
# 3. DỮ LIỆU: TẢI RAW -> LÀM SẠCH -> CHIA NHÓM -> EDA
assert not active_training(), 'Không đổi dữ liệu trong lúc train'
shell([PY,'scripts/download_data.py'],ROOT)
DATA=ROOT/'data/tensorflow_v1'
reference=ROOT/'data/reference_dataset_from_public_repo'
import shutil
if not reference.exists(): shutil.copytree(DATA,reference)
expected_manifest=sha(reference/'manifest.json')
if not (ROOT/'outputs/raw_rebuild_verified.json').exists():
    if DATA.exists():
        backup=ROOT/'data/before_raw_rebuild'
        assert not backup.exists(), 'Đã có lần prepare chưa kiểm chứng; cần rà soát'
        DATA.rename(backup)
    shell([PY,'-m','career_ai','prepare'],ROOT)
    assert sha(DATA/'manifest.json')==expected_manifest, 'Dataset tái tạo khác bản đóng băng; không train'
    (ROOT/'outputs/raw_rebuild_verified.json').write_text(json.dumps(dict(raw_downloaded=True,recreated=True,manifest_sha256=expected_manifest)))
manifest=json.loads((DATA/'manifest.json').read_text())
for split,meta in manifest['files'].items():
    assert sha(DATA/(split+'.csv'))==meta['sha256']
    print(split,meta['rows'],meta['markets'])
for name,checksum in manifest['artifacts'].items(): assert sha(DATA/name)==checksum,name
print(manifest['units'],manifest['note'])
shell([PY,'-c',"from career_ai.data import ROOT,load;from career_auto.charts import charts;charts(load('train'),ROOT/'outputs/eda_train')"],ROOT)
print('EDA: outputs/eda_train — khảo sát công nghệ đã dùng khác nhu cầu tuyển dụng.')


In [ ]:
# 4. TENSORFLOW LINEAR BASELINE
assert not active_training()
assert (ROOT/'outputs/raw_rebuild_verified.json').exists()
baseline=ROOT/'outputs/colab_baseline'
if not baseline.exists():
    shell([PY,'-u','-m','career_ai','train-baseline','--run',baseline,'--epochs','100'],ROOT)
else:
    assert json.loads((baseline/'lineage.json').read_text())['status']=='completed','Rà soát baseline đang lỗi'
shell([PY,'scripts/check_saved_run.py','--model','baseline','--run',baseline],ROOT)
shell([PY,'-m','career_auto.evaluation','--model','baseline','--run',baseline,'--split','valid'],ROOT)
print((baseline/'baseline_valid_metrics.json').read_text())


In [ ]:
# 5. TẢI PRETRAINED QWEN3 CHÍNH THỨC
# Model 1.7B, TensorFlow/KerasHub, tải đúng revision. Không nạp adapter Mac.
assert not active_training(), 'Không tải/sửa pretrained trong lúc train'
shell([PY,'scripts/download_qwen.py'],ROOT)
source=json.loads((ROOT/'configs/qwen_source.json').read_text())
assert sha(ROOT/source['local_path']/'model.weights.h5')==source['sha256']
print('Verified:',source['repo'],source['revision'])


In [ ]:
# 6. KIỂM TRA A100 VÀ CÀI CLI/BIỂU ĐỒ ĐÚNG GOAL
assert not active_training()
info=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True)
print(info)
assert 'A100' in info,'Dừng: chưa được cấp A100; không tự chuyển training sang CPU'
# Giữ nguyên trainer FP32 gốc: cả backbone và language loss đều trên GPU.
trainer=ROOT/'career_auto/training.py'
expected=subprocess.check_output(['git','show',COMMIT+':career_auto/training.py'],cwd=ROOT,text=True)
if trainer.read_text()!=expected:
    backup=ROOT/'outputs/training_before_a100_restore.py'
    backup.write_text(trainer.read_text())
    trainer.write_text(expected)
assert 'with tf.device("/CPU:0")' not in trainer.read_text()
shell([PY,'-c',"from career_auto.state import FOLDER;FOLDER.mkdir(parents=True,exist_ok=True);from career_auto.coordinator import install_validation_default,install_analysis_charts;state={};install_validation_default(state);install_analysis_charts(state);print(state)"],ROOT)
print('FP32 GPU training; context1024; không offload language loss sang CPU.')


In [ ]:
# 7. SMOKE THẬT: TRAIN -> CHECKPOINT -> RESUME -> NẠP LẠI
assert not active_training(), 'Đang train, không chạy thêm smoke'
config=ROOT/'configs/colab_a100_smoke.json'
config.write_text(json.dumps(dict(rank=8,learning_rate=5e-5,epochs=1,limit=20)))
smoke=ROOT/'outputs/colab_a100_smoke'
if not smoke.exists():
    train(config,smoke,['--checkpoint-stop-after','2'])
if json.loads((smoke/'lineage.json').read_text())['status']!='completed':
    train(config,smoke,['--resume'])
shell([PY,'-m','career_auto.verify_smoke','--run',smoke],ROOT)
print((smoke/'smoke_acceptance.json').read_text())
# Smoke chứng minh cập nhật/resume/lưu-nạp, không phải chất lượng dự đoán.


In [ ]:
# 8. TRAIN FULL TỪ PRETRAINED: TỐI ĐA 2 EPOCH
# rank8, lr5e-5, context1024, microbatch1, accumulation8; giữ toàn bộ train split.
assert not active_training(), 'Đã có training đang chạy!'
smoke=ROOT/'outputs/colab_a100_smoke'
assert json.loads((smoke/'smoke_acceptance.json').read_text())['fresh_process_prediction_parity']
full_config=ROOT/'configs/colab_a100_full.json'
expected=dict(rank=8,learning_rate=5e-5,epochs=2)
if full_config.exists(): assert json.loads(full_config.read_text())==expected
else: full_config.write_text(json.dumps(expected))
full=ROOT/'outputs/colab_a100_full'
if not full.exists():
    train(full_config,full)
elif json.loads((full/'lineage.json').read_text())['status']!='completed':
    train(full_config,full,['--resume'])
else:
    print('Đã hoàn thành; không train lại.')


In [ ]:
# 9. LƯU/NẠP VÀ ĐÁNH GIÁ VALIDATION — CHƯA MỞ TEST
assert not active_training(), 'Chờ model hoàn thành'
full=ROOT/'outputs/colab_a100_full'
assert json.loads((full/'lineage.json').read_text())['status']=='completed'
shell([PY,'scripts/check_saved_run.py','--model','llm','--run',full],ROOT)
shell([PY,'-m','career_auto.evaluation','--run',full,'--split','valid'],ROOT)
print((full/'llm_history.json').read_text())
print((full/'llm_valid_metrics.json').read_text())


In [ ]:
# SO SÁNH VALIDATION: LINEAR VÀ LLM, MAE/RMSE/R² RIÊNG TỪNG THỊ TRƯỜNG
assert not active_training()
full=ROOT/'outputs/colab_a100_full'
baseline=ROOT/'outputs/colab_baseline'
rows=[]
for family,run in [('baseline',baseline),('llm',full)]:
    metrics=json.loads((run/(family+'_valid_metrics.json')).read_text())
    rows.extend(dict(model=family,market=market,**values) for market,values in metrics['by_market'].items())
for row in rows: print(row)
(ROOT/'outputs/validation_comparison.json').write_text(json.dumps(rows,indent=2))
# Không xem/sử dụng test để chỉnh model; lỗi lớn và LLM thua baseline vẫn giữ nguyên.


In [ ]:
# DEMO ĐẦY ĐỦ: CSV/XLSX/JSON -> DỰ ĐOÁN -> BIỂU ĐỒ -> TƯ VẤN
assert not active_training()
full=ROOT/'outputs/colab_a100_full'
(ROOT/'outputs/auto_dev').mkdir(parents=True,exist_ok=True)
shell([PY,'-m','career_auto.demo','--run',full],ROOT)
print((ROOT/'outputs/auto_dev/demo/acceptance.json').read_text())
print((full/'advice.json').read_text()[:6000])
# Kiểm tra 3 định dạng, lỗi từng dòng, đơn vị, phiên bản model, kỹ năng có bằng chứng.
# Ví dụ suy luận file mới (không train):
# shell([PY,'-m','career_ai','analyze','--run',full,'--input','FILE.csv'],ROOT)
# shell([PY,'-m','career_ai','advise','--run',full,'--input','FILE.json'],ROOT)


In [ ]:
# TEST CHỈ CHO CHECKPOINT CUỐI ĐÃ CHỌN BẰNG VALIDATION
assert not active_training()
full=ROOT/'outputs/colab_a100_full'
baseline=ROOT/'outputs/colab_baseline'
assert json.loads((full/'lineage.json').read_text())['status']=='completed'
assert (ROOT/'outputs/validation_comparison.json').exists()
assert json.loads((ROOT/'outputs/auto_dev/demo/acceptance.json').read_text())['passed']
state=dict(status='final_evaluation',selected_run=str(full.relative_to(ROOT)),test_authorized=True,final_step=4)
(ROOT/'outputs/auto_dev/state.json').write_text(json.dumps(state,indent=2))
for family,run in [('baseline',baseline),('llm',full)]:
    output=run/(family+'_test_metrics.json')
    if not output.exists():
        shell([PY,'-m','career_auto.evaluation','--model',family,'--run',run,'--split','test'],ROOT)
# Nguồn đã được nghiên cứu trước; không gọi đây là benchmark mù mới.
comparison=ROOT/'outputs/auto_dev/comparison'
if not comparison.exists():
    shell([PY,'scripts/compare_runs.py','--baseline',baseline,'--llm',full,'--output',comparison],ROOT)
print((comparison/'comparison.csv').read_text())


In [ ]:
# KIỂM THỬ, 5 NOTEBOOK KERNEL SẠCH VÀ MANIFEST BÀN GIAO
assert not active_training()
shell([PY,'-m','unittest','discover','-s','tests','-v'],ROOT)
policy_path=ROOT/'configs/auto_goal.json'
policy=json.loads(policy_path.read_text())
policy['baseline_run']='outputs/colab_baseline'
policy['current_run']='outputs/colab_a100_full'
policy_path.write_text(json.dumps(policy,indent=2))
shell([PY,'-m','ipykernel','install','--prefix','/content/career_tf_env','--name','career-ai-tf','--display-name','Career AI TensorFlow Colab'],ROOT)
shell([PY,'-m','career_auto.notebook_handoff'],ROOT)
artifacts={str(p.relative_to(ROOT)):sha(p) for folder in [ROOT/'outputs',ROOT/'notebooks'] for p in folder.rglob('*') if p.is_file() and p.name!='COLAB_HANDOFF_MANIFEST.json'}
(ROOT/'outputs/COLAB_HANDOFF_MANIFEST.json').write_text(json.dumps(dict(files=artifacts,data_manifest=sha(ROOT/'data/tensorflow_v1/manifest.json'),source_commit=COMMIT,local_upload=False),indent=2))
print('Đã chạy kiểm thử và notebook; xem metric thật trong outputs. Không mặc định LLM tốt hơn baseline.')


In [ ]:
# 14. XEM TIẾN ĐỘ VÀ TẢI KẾT QUẢ VỀ
# /content là tạm thời: tải checkpoint trước khi VM bị xóa.
print('Tiến trình:', active_training())
for run in ['colab_a100_smoke','colab_a100_full']:
    for name in ['progress.json','llm_history.json','lineage.json']:
        path=ROOT/'outputs'/run/name
        if path.exists(): print(run,name,path.read_text()[:3000])
DOWNLOAD=False  # Đổi True khi muốn tải bản sao outputs/checkpoint.
if DOWNLOAD:
    assert not active_training(), 'Chờ checkpoint ổn định trước khi đóng gói'
    import shutil
    from google.colab import files
    files.download(shutil.make_archive('/content/colab_training_results','zip',ROOT,'outputs'))
